<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/EmbeddingsAnalysis_TopicModelling_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Clustring

### Clustering Topic Models from TurfTopic

1. TurfTopic Default model and configuration
2. Use other models like ModernBert

In [1]:
# Install libraries and packages
!pip install 'turftopic[umap-learn, datamapplot]'

In [2]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [4]:
# Read and Load dataset
dataset = pd.read_csv('sample_PubMedDataAbstracts.csv')

# Show the datasets
### Abstract Embeddings Sample Dataset
print('Node Content:', dataset.shape)
print(dataset)

Node Content: (10000, 4)
      Unnamed: 0                                              title  \
0              0  Phenotypic variability of Niemann-Pick disease...   
1              1  Recurrent hypoglycemia secondary to metformin ...   
2              2  Adaptation of the Ambulatory and Home Care Rec...   
3              3  Multidimensional family therapy in adolescents...   
4              4  Balanced crystalloids versus isotonic saline i...   
...          ...                                                ...   
9995        9995  Methylmercury in Industrial Harbor Sediments i...   
9996        9996  Factors Affecting Secondhand Smoke Avoidance B...   
9997        9997  Predicting Infectious Disease Using Deep Learn...   
9998        9998  Diosgenin Glucoside Protects against Spinal Co...   
9999        9999  Omics Approaches for Engineering Wheat Product...   

                                               abstract  year  
0     Background Niemann-Pick disease type C (NPC) i...  2

In [5]:
# Extract only the 'abstract' column and drop others
abstracts = dataset['abstract'].dropna().reset_index(drop=True)

# Display a few samples to verify
print(abstracts)

0       Background Niemann-Pick disease type C (NPC) i...
1       Background Metformin toxicity is well known to...
2       Background Measuring service use and costs is ...
3       Background Substance use and delinquency are c...
4       Objectives Intravenous fluids are one of the m...
                              ...                        
9995    The distribution of methylmercury (MeHg) and t...
9996    The purpose of this study was to examine the s...
9997    Infectious disease occurs when a person is inf...
9998    Spinal cord injury (SCI) is a severe traumatic...
9999    Abiotic stresses greatly influenced wheat prod...
Name: abstract, Length: 10000, dtype: object


### 1. TurfTopic Default model and configuration

In [6]:
# Import topic clustring required libraries
from sentence_transformers import SentenceTransformer
from turftopic import Top2Vec

In [ ]:
# Using TurfTopic default encoder to extract embedding of the dataset
encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(abstracts, show_progress_bar=True)

In [14]:
# Show embeddings matrix and Check the dimention of each eambeding
print(embeddings,"\n\n", embeddings.shape)

[[-0.06247491 -0.06421211 -0.04776807 ...  0.05779283 -0.00046217
  -0.0348533 ]
 [ 0.03635849 -0.05012973 -0.0077231  ... -0.00230364  0.01381705
  -0.08019841]
 [ 0.01247236  0.01453279 -0.01938152 ...  0.00787823  0.01995303
  -0.02415804]
 ...
 [-0.00543451 -0.08347747  0.03953594 ... -0.01684228 -0.10415071
   0.11270402]
 [-0.06943101 -0.0625765  -0.00057785 ...  0.00205806  0.01154448
  -0.04431521]
 [-0.07437815  0.00144461 -0.0106479  ... -0.01505966 -0.03949417
  -0.08860712]] 

 (10000, 384)


In [15]:
# Training model (Uses HDBSCAN and umap)
model = Top2Vec(encoder=encoder, random_state=42)
topic_data = model.prepare_topic_data(abstracts, embeddings=embeddings)

Output()

[11:07:04] Term extraction done.                                                                     ]8;id=455768;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=558947;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py#443\443]8;;\

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was 
renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(

/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting 
random_state. Use no seed for parallelism.
  warn(

[11:07:49] Dimensionality reduction done.                                                            ]8;id=288736;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=894845;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py#448\448]8;;\

[11:07:50] Clustering done.                                                                          ]8;id=929005;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=431795;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py#451\451]8;;\

[11:07:53] Parameter estimation done.                                                                ]8;id=811673;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=813232;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py#470\470]8;;\

           Model fitting done.                                                                       ]8;id=473378;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=945001;file:///usr/local/lib/python3.11/dist-packages/turftopic/models/cluster.py#484\484]8;;\

In [19]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ metabolomic, metabolomics, biomarkers, proteomics, proteomic, pharmacologically, pharmacological,    │
│          │ biomarker, pharmacology, metabolome                                                                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ retinopathy, retinal, intraocular, glaucoma, cataract, ocular, retina, ophthalmic, corneal, macular  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ implant, implants, osteoclasts, osteoblasts, osteogenic, osteoclast, implantation, implanted, bone,  │
│          │ osteoblast                                                                                           │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ dental, tooth, oral, teeth, periodontitis, periodontal, dent, caries, orally, hygiene                │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ periodontitis, periodontal, gingival, dental, cytokine, oral, inflammation, cytokines, tooth,        │
│          │ interleukin                                                                                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ dental, teeth, tooth, periodontal, dent, enamel, periodontitis, maxillary, resin, oral               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ judgments, integrity, ethics, inherent, normative, judgment, moral, interpretation, interpretations, │
│          │ necessitates                                                                                         │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        6 │ pollutants, epidemiology, pollution, epidemiological, epidemiologic, pollutant, asthma, polluted,    │
│          │ toxicity, inhaled                                                                                    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        7 │ microscopy, microscope, fluorescence, imaging, immunofluorescence, subcellular, proteomics,          │
│          │ intracellular, intercellular, nanoscale                                                              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │ tuberculosis, mycobacterium, mycobacterial, tb, antibacterial, bactericidal, pathogen, bacteremia,   │
│          │ pneumococcal, pathogens                                                                              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        9 │ classifiers, classifying, classification, classify, supervised, classifier, cnn, bioinformatics,     │
│          │ biomarker, datasets                                                                                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │ ligands, ligand, compounds, synthesized, molecule, synthesis, catalysis, molecular, aromatic,        │
│          │ catalyzed                                  

In [26]:
# Cluster model hierarchy
model.hierarchy.cut(3).plot_tree()

In [27]:
# Merging topics to reduce number of topics
model.reduce_topics(n_reduce_to=25)
print(model.hierarchy.cut(3))

/usr/local/lib/python3.11/dist-packages/turftopic/models/_hierarchical_clusters.py:260: UserWarning:

Number of clusters is already 25 <= 25, nothing to do.



Root: 
├── -1: metabolomic, metabolomics, biomarkers, proteomics, proteomic, pharmacologically, 
│   pharmacological, biomarker, 
│   pharmacology, metabolome
├── 11: sensor, sensing, sensors, accelerometers, gps, accelerometer, iot, tracking, monitoring, 
│   802
├── 104: species, speciation, ecology, phenotypic, phylogenies, biodiversity, fauna, phylogenetic, 
│   taxonomic, 
│   ecological
│   ├── 42: pollination, flowering, plants, pollen, floral, phenotypic, botanical, speciation, flower,
│   │   cultivars
│   └── 43: species, ecology, speciation, fauna, phylogenies, biodiversity, phenotypic, phylogenetic,
│       taxonomic, ecological
├── 113: phytochemical, phytochemicals, flavonoids, antioxidants, antioxidant, antioxidative, 
│   metabolites, phenolics, 
│   flavonoid, herbal
│   ├── 52: phytochemicals, metabolites, antimicrobials, phytochemical, alkaloids, bioactivities, 
│   │   alkaloid, biosynthesis, 
│   │   biogeochemical, biochemical
│   └── 53: phytochemical, phytochemi

In [28]:
# Model hierarchy after merging topics
fig = model.hierarchy[170].plot_tree()
fig.show()